# Limpeza De dados 

## Bibliotecas Utilizadas

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import re

## Funções 

### Funções utilizadas no tratamento dos dados

In [2]:
pattern_process_anatel = re.compile(r'\b\d{5}\.\d{6}/(?:19|20)\d{2}-\d{2}\b')
cnpj_pattern = re.compile(
    r'\b\d{2}[\.\s/-]?\d{3}[\.\s/-]?\d{3}[\.\s/-]?\d{4}[\.\s/-]?\d{2}\b'
)


def normalizar_texto(texto:str)->str:
    if pd.isna(texto):
        return texto
    novo_texto = str(texto).strip()
    novo_texto = re.sub(r'\s+', ' ', novo_texto)
    return novo_texto

def filtrar_processos_validos(texto:str)->str:
    if pd.isna(texto):
        return np.nan
    matches = pattern_process_anatel.findall(texto)
    if len(matches) >= 1:
        return matches[0]
    else: 
        return np.nan
    

def extract_valid_cnpjs(texto:str)->str:
    if pd.isna(texto): 
        return texto
    texto = str(texto)
    candidatos = cnpj_pattern.findall(texto)
    if len(candidatos) > 0:
        numeros = re.sub(r'\D', '', candidatos[0])
        return numeros
    return np.nan

def format_cnpj(texto_cnpj:str)->str:
    if pd.isna(texto_cnpj): 
        return texto_cnpj
    texto_cnpj = str(texto_cnpj)
    cnpj = re.sub(r'\D', '', texto_cnpj)

    if len(cnpj) != 14:
        return np.nan

    return (
        f"{cnpj[:2]}.{cnpj[2:5]}.{cnpj[5:8]}/{cnpj[8:12]}-{cnpj[12:]}"
    )

    

# def limpar_texto(texto):
#     if pd.isna(texto):
#         return texto
    
#     texto = str(texto).lower()
    
#     texto = unicodedata.normalize('NFD', texto)
#     texto = ''.join(
#         c for c in texto
#         if unicodedata.category(c) != 'Mn'
#     )

#     texto = re.sub(r'[^a-z0-9\s]', '', texto)

#     texto = re.sub(r'\s+', ' ', texto).strip()

#     return texto

In [3]:
def limpar_texto(texto):
    if pd.isna(texto):
        return texto
    
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(
        c for c in texto
        if unicodedata.category(c) != 'Mn'
    )

    texto = re.sub(r'[^a-z0-9\s]', '', texto)

    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto

def limpeza_sei(texto):
    if pd.isna(texto):
        return texto
    
    texto = str(texto).lower()
    
    if texto.find('/') > -1: 
        texto = texto[:texto.find('/')]
    if texto.find(' ') > -1:
        texto = texto[:texto.find(' ')]
    texto = re.sub(r'[a-zA-Z///*]', '', texto).strip()
    if not texto: 
        return np.nan
    else: 
        return texto





def tratar_strings_df(df, exceptions:list[str]=[]):
    df = df.copy()

    colunas_texto = df.select_dtypes(include=['object', 'string']).columns

    for coluna in colunas_texto:
        if coluna not in exceptions:
            df[coluna] = df[coluna].apply(limpar_texto)

    return df

def coluna_data(df, list_columns):
    for col in df.columns:
        if col in list_columns:
            df[col] = pd.to_datetime(
                df[col],
                format='%d/%m/%Y',
                errors='coerce'
            )


    return df

In [4]:
sei_invalido = [
    'NÃO APLICÁVEL',
    'COM PENDÊNCIA',
    'DESNECESSÁRIO',
    'NÃO INFORMADO',
    'NÃO SE APLICA',
    'PRESTAÇÃO IRREGULAR',
    'NÃO HOUVE',
    'ARQUIVO NÃO ABRE',
]

modalidades_stfc = [
    'LDI', 'LDN', 'LOCAL'
]
modalidades_stfc_invalida = 'INDEFINIDA'

## Limpeza de Dataset 

### Credenciadas Vigentes 



In [5]:
df_empresas_cred = pd.read_csv(
    r'Dados - Contratos/empresas_credenciadas_vigentes.csv',
    sep=';'
)

print(df_empresas_cred.shape)

(196, 2)


### Contratos - Compartilhamento

Esse dataset é sobre os Contratos de Compartilhamento de infraestrutura passiva entre concessionárias de energia elétrica (detentoras de postes, dutos, torres) e operadoras de telecom


#### Leitura do Dataset 

In [70]:
df_contratos_comp = pd.read_csv(
    r'Dados - Contratos/contratos_compartilhamento.csv',
    sep = ';'
)

print(df_contratos_comp.shape)
df_contratos_comp.head(5)

(9630, 11)


,ACORDO_TIPO,PROTOCOLO_DATA,PROCESSO_ANATEL,DETENTORA,DETENTORA_CNPJ,SOLICITANTE,SOLICITANTE_CNPJ,INFORME_SEI,INFORME_DATA,CONCLUSAO_DATA,OBSERVACAO
0,CONTRATO,22/04/2013,53500.009204/2013-64,LIGHT SERVIÇOS DE ELETRICIDADE S.A,6.044444e+13,AFINET,NaN,70/2013-CPRP/SCP,17/06/2013,17/06/2013,Processo ANEEL: 48500.002273/2013-15. Despacho...
1,CONTRATO,02/05/2013,53500.011549/2013-88,CAIUÁ - DISTRIBUIÇÃO DE ENERGIA S.A.,7.282377e+12,ACER TELECOMUNICAÇÕES,NaN,66/2013-CPRP/SCP,14/06/2013,17/06/2013,Processo ANEEL: 48500.001859/2013-54. \r
2,CONTRATO,06/05/2013,53500.011548/2013-33,CAIUÁ - DISTRIBUIÇÃO DE ENERGIA S.A.,7.282377e+12,NOVA PORTONET,NaN,120/2013-CPRP/SCP,28/06/2013,03/07/2013,Processo ANEEL: 48500.001858/2013-18. Despacho...
3,CONTRATO,06/05/2013,53500.012976/2013-83,COPEL DISTRIBUIÇÃO S.A.,4.368898e+12,CIABRASNET,NaN,73/2013-CPRP/SCP,18/06/2013,18/06/2013,Processo ANEEL: 48500.002187/2013-02. \r
4,CONTRATO,06/05/2013,53500.012976/2013-83,COPEL DISTRIBUIÇÃO S.A.,4.368898e+12,DELTA TELECOMUNICAÇÕES,NaN,73/2013-CPRP/SCP,18/06/2013,18/06/2013,Processo ANEEL: 48500.002271/2013-18. Despacho...


#### Tratamento de Duplicas 

In [71]:
print(f'Tamanho dataset ANTES: {df_contratos_comp.shape}')
df_contratos_comp = df_contratos_comp.drop_duplicates()
print(f'Tamanho dataset DEPOIS: {df_contratos_comp.shape}')

Tamanho dataset ANTES: (9630, 11)
Tamanho dataset DEPOIS: (9619, 11)


####  Tratamento Colunas de Data 

In [72]:
colunas_data = ['PROTOCOLO_DATA', 'CONCLUSAO_DATA', 'INFORME_DATA']
df_contratos_comp[colunas_data].isna().sum()
for col in colunas_data:
    df_contratos_comp[col] = pd.to_datetime(
        df_contratos_comp[col],
        format='%d/%m/%Y',
        errors='coerce'
    )
df_contratos_comp[colunas_data].sample(5)

,PROTOCOLO_DATA,CONCLUSAO_DATA,INFORME_DATA
7213,2022-07-21,2022-09-02,2022-09-02
3463,2018-08-17,2018-09-19,2018-09-19
7541,2022-10-11,2022-11-11,2022-11-11
8134,2023-06-12,2023-08-02,2023-08-02
6050,2021-08-23,2021-10-28,2021-10-28


#### Tratamento de Colunas de TEXTO 

In [73]:
colunas_texto = df_contratos_comp.select_dtypes(include=['object', 'string']).columns
colunas_texto

Index(['ACORDO_TIPO', 'PROCESSO_ANATEL', 'DETENTORA', 'SOLICITANTE',
       'INFORME_SEI', 'OBSERVACAO'],
      dtype='str')

In [74]:
# Normalizando padrão textual do ACOROD_TIPO e preenchendo com "INDEFINIDO" para nulos
df_contratos_comp['ACORDO_TIPO'] = (
    df_contratos_comp['ACORDO_TIPO']
    .apply(normalizar_texto)
    .str.upper()
    .fillna('INDEFINIDO')
)

# Filtrando apenas registros de processos válidos 
df_contratos_comp['PROCESSO_ANATEL'] = (
    df_contratos_comp['PROCESSO_ANATEL']
        .apply(normalizar_texto)
        .apply(filtrar_processos_validos)
)


# Normalizando padrão textual do DETENTORA e preenchendo com "INDEFINIDO" para nulos
df_contratos_comp['DETENTORA'] = (
    df_contratos_comp['DETENTORA']
    .apply(normalizar_texto)
    .str.upper()
)
df_contratos_comp['SOLICITANTE'] = (
    df_contratos_comp['SOLICITANTE']
    .apply(normalizar_texto)
    .str.upper()
)


# Tratando a coluna de SEI, que muitas vezes apresenta informações parciais.
# Estamos também tratando algumas colunas erradas como vazias/nulas

df_contratos_comp['INFORME_SEI'] = (
    df_contratos_comp['INFORME_SEI']
    .apply(normalizar_texto)
    .str.upper()
    .map(lambda x: x if pd.isna(x) or str(x) not in sei_invalido else np.nan)
    .str.replace(' ', '')
    .astype(str)
)

# Normalizando os CNPJS da Detentora e da Solicitante. As vezes não teremos CNPJ preenchido, mas temos que ter
# ao menos o nome da empresa 

df_contratos_comp['SOLICITANTE_CNPJ'] = (
    df_contratos_comp['SOLICITANTE_CNPJ']
        .apply(normalizar_texto)
        .str.replace(' ', '')
        .apply(extract_valid_cnpjs)
        .apply(format_cnpj)
    )

df_contratos_comp['DETENTORA_CNPJ'] = (
    df_contratos_comp['DETENTORA_CNPJ']
        .apply(normalizar_texto)
        .str.replace(' ', '')
        .apply(extract_valid_cnpjs)
        .apply(format_cnpj)
    )

# O campo de Observações é uma área de texto llivre, vamos só tirar valores problematicos e manter null quando
# necessário 

df_contratos_comp['OBSERVACAO'] = (
    df_contratos_comp['OBSERVACAO']
        .str.strip()
        .map(lambda x: np.nan if pd.isna(x) or len(str(x)) == 0 else str(x))
        .astype(str)        
    )


# Selecionando apenas as linhas com processo anatel, Detentora e Solicitantes preenchidos 
df_contratos_comp = df_contratos_comp[
    (~df_contratos_comp['PROCESSO_ANATEL'].isna())  & 
    (~df_contratos_comp['SOLICITANTE'].isna())  & 
    (~df_contratos_comp['DETENTORA'].isna()) 
]
df_contratos_comp

,ACORDO_TIPO,PROTOCOLO_DATA,PROCESSO_ANATEL,DETENTORA,DETENTORA_CNPJ,SOLICITANTE,SOLICITANTE_CNPJ,INFORME_SEI,INFORME_DATA,CONCLUSAO_DATA,OBSERVACAO
0,CONTRATO,2013-04-22,53500.009204/2013-64,LIGHT SERVIÇOS DE ELETRICIDADE S.A,60.444.437/0001-46,AFINET,NaN,70/2013-CPRP/SCP,2013-06-17,2013-06-17,Processo ANEEL: 48500.002273/2013-15. Despacho...
1,CONTRATO,2013-05-02,53500.011549/2013-88,CAIUÁ - DISTRIBUIÇÃO DE ENERGIA S.A.,NaN,ACER TELECOMUNICAÇÕES,NaN,66/2013-CPRP/SCP,2013-06-14,2013-06-17,Processo ANEEL: 48500.001859/2013-54.
2,CONTRATO,2013-05-06,53500.011548/2013-33,CAIUÁ - DISTRIBUIÇÃO DE ENERGIA S.A.,NaN,NOVA PORTONET,NaN,120/2013-CPRP/SCP,2013-06-28,2013-07-03,Processo ANEEL: 48500.001858/2013-18. Despacho...
3,CONTRATO,2013-05-06,53500.012976/2013-83,COPEL DISTRIBUIÇÃO S.A.,NaN,CIABRASNET,NaN,73/2013-CPRP/SCP,2013-06-18,2013-06-18,Processo ANEEL: 48500.002187/2013-02.
4,CONTRATO,2013-05-06,53500.012976/2013-83,COPEL DISTRIBUIÇÃO S.A.,NaN,DELTA TELECOMUNICAÇÕES,NaN,73/2013-CPRP/SCP,2013-06-18,2013-06-18,Processo ANEEL: 48500.002271/2013-18. Despacho...
...,...,...,...,...,...,...,...,...,...,...,...
9625,CONTRATO,2025-04-07,53500.025708/2025-65,AMAZONAS ENERGIA S.A.,NaN,UPDATA TELECOMUNICAÇÕES E INFORMÁTICA LTDA,57.133.512/0001-43,13722855,2025-05-27,2025-05-30,NaN
9626,CONTRATO,2025-04-15,53500.028415/2025-30,AMAZONAS ENERGIA S.A.,NaN,NET ACCESS TELECOMUNICAÇÕES LTDA,28.606.897/0001-10,13722855,2025-05-27,2025-05-30,NaN
9627,CONTRATO,2025-04-16,53500.028619/2025-71,COOPERATIVA DE ELETRIFICAÇÃO E DESENVOLVIMENTO...,53.176.038/0001-86,VOX SOLUÇÕES EM INTERNET,28.488.672/0001-07,13722855,2025-05-27,2025-05-30,NaN
9628,CONTRATO,2025-04-29,53500.031783/2025-65,AMAZONAS ENERGIA S.A.,NaN,T G L SERVIÇOS DE TELECOMUNICAÇÕES LTDA,28.450.243/0001-40,13722855,2025-05-27,2025-05-30,NaN


#### Remoção de Duplicadas (novamente )

In [75]:
print(f'Tamanho dataset ANTES: {df_contratos_comp.shape}')
df_contratos_comp = df_contratos_comp.drop_duplicates()
print(f'Tamanho dataset DEPOIS: {df_contratos_comp.shape}')

Tamanho dataset ANTES: (9617, 11)
Tamanho dataset DEPOIS: (9617, 11)


#### Coluna de versionamento 

Vammos criar uma coluna para salvar as informações de versionamento de um contrato: 

* `Num Sequencia`, qual é a versão que aquele contrato está
* `final`, se aquela é a versão final ou não. 



In [77]:
df_contratos_comp = (
    df_contratos_comp
        .sort_values(by=['PROCESSO_ANATEL', 'PROTOCOLO_DATA'], kind='stable')
)
df_contratos_comp['NUM_SEQUENCIA'] = df_contratos_comp.groupby(by='PROCESSO_ANATEL').cumcount() + 1
df_max_seq = (
    df_contratos_comp
        .groupby('PROCESSO_ANATEL')
        [['NUM_SEQUENCIA']]
        .max()
        .reset_index()
        .rename(columns={'NUM_SEQUENCIA': 'MAX_NUM_SEQ'})
)
df_contratos_comp = df_contratos_comp.merge(df_max_seq, on='PROCESSO_ANATEL', how='left')
df_contratos_comp['FINAL']  = df_contratos_comp['NUM_SEQUENCIA'] == df_contratos_comp['MAX_NUM_SEQ']
df_contratos_comp = df_contratos_comp.drop(columns=['MAX_NUM_SEQ'])
df_contratos_comp

,ACORDO_TIPO,PROTOCOLO_DATA,PROCESSO_ANATEL,DETENTORA,DETENTORA_CNPJ,SOLICITANTE,SOLICITANTE_CNPJ,INFORME_SEI,INFORME_DATA,CONCLUSAO_DATA,OBSERVACAO,NUM_SEQUENCIA,FINAL
0,CONTRATO,2013-06-12,48513.023648/2015-95,CELESC DISTRIBUIÇÃO S.A.,NaN,SUL AMERICANA TECNOLOGIA E INFORMÁTICA LTDA.,NaN,01/2016/CPRP/SCP,2016-01-08,2016-01-08,Processo ANEEL: 48513.023648/2015-00.,1,True
1,CONTRATO,2015-01-23,48526.000122/2015-98,AES SUL DISTRIBUIDORA GAÚCHA DE ENERGIA S.A.,NaN,T.C.A INFORMATICA LTDA.,NaN,2452,2015-03-04,2015-03-09,Processo Aditivo: 48526.000122/2015. Processo ...,1,False
2,CONTRATO,2015-01-23,48526.000122/2015-98,COPEL DISTRIBUIÇÃO S.A.,NaN,COMFIBRA - PROVEDOR DE TELECOMUNICAÇÕES LTDA.,NaN,2452,2015-03-04,2015-03-09,Processo Aditivo: 48526.000122/2015. Processo ...,2,False
3,CONTRATO,2015-01-23,48526.000122/2015-98,COPEL DISTRIBUIÇÃO S.A.,NaN,DIRECT WIFI TELECOM LTDA.,NaN,2452,2015-03-04,2015-03-09,Processo Aditivo: 48526.000122/2015. Processo ...,3,False
4,CONTRATO,2015-01-23,48526.000122/2015-98,COPEL DISTRIBUIÇÃO S.A.,NaN,ANTONIO MARCOS CRUZ & CIA LTDA.,NaN,2452,2015-03-04,2015-03-09,Processo Aditivo: 48526.000122/2015. Processo ...,4,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9612,CONTRATO,2019-02-04,53569.000211/2019-00,CENTRAIS ELÉTRICAS DO PARÁ S/A - CELPA,NaN,COLLERE SERVIÇOS DE TELECOM. LTDA - ME,83.337.030/0001-15,4238980,2019-06-07,2019-06-07,NaN,4,False
9613,CONTRATO,2019-02-04,53569.000211/2019-00,CENTRAIS ELÉTRICAS DO PARÁ S/A - CELPA,NaN,ITBNET PROVEDOR LTDA EPP,NaN,4238980,2019-06-07,2019-06-07,NaN,5,False
9614,CONTRATO,2019-02-04,53569.000211/2019-00,CENTRAIS ELÉTRICAS DO PARÁ S/A - CELPA,NaN,J E RODRIGUES DE OLIVEIRA - ME,NaN,4238980,2019-06-07,2019-06-07,NaN,6,False
9615,CONTRATO,2019-02-04,53569.000211/2019-00,CENTRAIS ELÉTRICAS DO PARÁ S/A - CELPA,NaN,JC TELECOM SERVIÇOS DE TELECOMUNICAÇÕES LTDA -...,NaN,4238980,2019-06-07,2019-06-07,NaN,7,False


### Contratos - Interconexão 

#### Leitura do Dataset 

In [12]:
df_contratos_int = pd.read_csv(
    r'Dados - Contratos/contratos_interconexao.csv',
    sep=';'
)

print(df_contratos_int.shape)
df_contratos_int.head(5)


(12374, 15)


,PROCESSO_ANATEL,PROTOCOLO_DATA,ACORDO_TIPO,PRESTADORA_1_CNPJ,PRESTADORA_1,SERVICO_1,MODALIDADE_STFC_1,PRESTADORA_2_CNPJ,PRESTADORA_2,SERVICO_2,MODALIDADE_STFC_2,DESPACHO_DECISORIO_SEI,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA,OBSERVACAO
0,53500.028940/2006-92,26/10/2006,NÃO INFORMADO,2449992000164,VIVO S.A.,10,NaN,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,20,NaN,-1,06/10/2014,06/10/2014,Classe: IV. Localização: ARQUIVO GERAL.
1,53500.004342/2007-17,14/02/2007,CONTRATO,2449992000164,VIVO S.A.,10,NaN,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,20,NaN,47,05/08/2015,19/08/2015,Classe: IV. Localização: ESTAÇÃO.
2,53500.021964/2008-82,14/08/2008,NÃO INFORMADO,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,10,NaN,5958690000100,UNICEL DO BRASIL TELECOMUNICAÇÕES LTDA.,10,NaN,3,13/06/2013,17/06/2013,Classe: IV. Localização: ARQUIVO SETORIAL.
3,53500.004720/2009-16,04/03/2009,NÃO INFORMADO,2449992000164,VIVO S.A.,10,NaN,5958690000100,UNICEL DO BRASIL TELECOMUNICAÇÕES LTDA.,10,NaN,2,13/06/2013,17/06/2013,Classe: IV. Localização: ARQUIVO SETORIAL.
4,53500.007610/2009-14,19/03/2009,NÃO INFORMADO,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,20,NaN,1009876000161,AGERA TELECOMUNICAÇÕES S.A.,20,NaN,4998,26/09/2014,29/09/2014,Classe: IV. Localização: ESTAÇÃO.


In [13]:
df_contratos_int.columns

Index(['PROCESSO_ANATEL', 'PROTOCOLO_DATA', 'ACORDO_TIPO', 'PRESTADORA_1_CNPJ',
       'PRESTADORA_1', 'SERVICO_1', 'MODALIDADE_STFC_1', 'PRESTADORA_2_CNPJ',
       'PRESTADORA_2', 'SERVICO_2', 'MODALIDADE_STFC_2',
       'DESPACHO_DECISORIO_SEI', 'DESPACHO_DECISORIO_DATA', 'CONCLUSAO_DATA',
       'OBSERVACAO'],
      dtype='str')

#### Tratamento de Duplicas 

In [14]:
print(f'Tamanho dataset ANTES: {df_contratos_int.shape}')
df_contratos_int = df_contratos_int.drop_duplicates()
print(f'Tamanho dataset DEPOIS: {df_contratos_int.shape}')

Tamanho dataset ANTES: (12374, 15)
Tamanho dataset DEPOIS: (12364, 15)


####  Tratamento Colunas de Data 

In [15]:
colunas_data = ['PROTOCOLO_DATA',  'CONCLUSAO_DATA', 'DESPACHO_DECISORIO_DATA']
df_contratos_int[colunas_data].isna().sum()
for col in colunas_data:
    df_contratos_int[col] = pd.to_datetime(
        df_contratos_int[col],
        format='%d/%m/%Y',
        errors='coerce'
    )
df_contratos_int[colunas_data].sample(5)

,PROTOCOLO_DATA,CONCLUSAO_DATA,DESPACHO_DECISORIO_DATA
7950,2021-05-18,2021-10-14,2021-10-05
5331,2020-05-14,2020-11-04,2020-11-04
5606,2020-07-01,2020-11-10,2020-11-10
7988,2021-05-18,2021-10-14,2021-10-05
1485,2015-05-12,2015-08-27,2015-08-21


#### Tratamento de Colunas de TEXTO 

In [16]:
colunas_texto = df_contratos_int.select_dtypes(include=['object', 'string']).columns
colunas_texto

Index(['PROCESSO_ANATEL', 'ACORDO_TIPO', 'PRESTADORA_1', 'MODALIDADE_STFC_1',
       'PRESTADORA_2', 'MODALIDADE_STFC_2', 'DESPACHO_DECISORIO_SEI',
       'OBSERVACAO'],
      dtype='str')

In [17]:
# Normalizando padrão textual do ACOROD_TIPO e preenchendo com "INDEFINIDO" para nulos
df_contratos_int['ACORDO_TIPO'] = (
    df_contratos_int['ACORDO_TIPO']
    .apply(normalizar_texto)
    .str.upper()
    .fillna('INDEFINIDO')
)

# Filtrando apenas registros de processos válidos 
df_contratos_int['PROCESSO_ANATEL'] = (
    df_contratos_int['PROCESSO_ANATEL']
        .apply(normalizar_texto)
        .apply(filtrar_processos_validos)
)


# Normalizando padrão textual do DETENTORA e preenchendo com "INDEFINIDO" para nulos
df_contratos_int['PRESTADORA_1'] = (
    df_contratos_int['PRESTADORA_1']
    .apply(normalizar_texto)
    .str.upper()
)
df_contratos_int['PRESTADORA_2'] = (
    df_contratos_int['PRESTADORA_2']
    .apply(normalizar_texto)
    .str.upper()
)

# Normalizando os CNPJS da PRESTADORA 1 e da PRESTADORA 2. As vezes não teremos CNPJ preenchido, mas temos que ter
# ao menos o nome da empresa 

df_contratos_int['PRESTADORA_1_CNPJ'] = (
    df_contratos_int['PRESTADORA_1_CNPJ']
        .apply(normalizar_texto)
        .str.replace(' ', '')
        .apply(extract_valid_cnpjs)
        .apply(format_cnpj)
    )

df_contratos_int['PRESTADORA_2_CNPJ'] = (
    df_contratos_int['PRESTADORA_2_CNPJ']
        .apply(normalizar_texto)
        .str.replace(' ', '')
        .apply(extract_valid_cnpjs)
        .apply(format_cnpj)
    )

# Limpando DESPACHO DECISORIO SEI, um valor INTEIRO mas as vezes vem com texto sujo ou nulo.
df_contratos_int['DESPACHO_DECISORIO_SEI'] = (
    df_contratos_int['DESPACHO_DECISORIO_SEI']
    .apply(normalizar_texto)
    .apply(limpeza_sei)
)

## Vamos limpar as modalidades STFC do Dataset. Quando nulas vamos preencher com: 'INDEFINIDA'.DS_Store

df_contratos_int['MODALIDADE_STFC_1'] = (
    df_contratos_int['MODALIDADE_STFC_1']
    .apply(normalizar_texto)
    .str.upper()
    .map(lambda x: np.nan if pd.isna(x) or str(x) not in modalidades_stfc else x)
    .fillna(modalidades_stfc_invalida)
    .astype(str)
)
df_contratos_int['MODALIDADE_STFC_2'] = (
    df_contratos_int['MODALIDADE_STFC_2']
    .apply(normalizar_texto)
    .str.upper()
    .map(lambda x: np.nan if pd.isna(x) or str(x) not in modalidades_stfc else x)
    .fillna(modalidades_stfc_invalida)
    .astype(str)
)


# O campo de Observações é uma área de texto llivre, vamos só tirar valores problematicos e manter null quando
# necessário 
df_contratos_int['OBSERVACAO'] = (
    df_contratos_int['OBSERVACAO']
        .str.strip()
        .map(lambda x: np.nan if pd.isna(x) or len(str(x)) == 0 else str(x))
        .astype(str)        
    )


# # Selecionando apenas as linhas com processo anatel, Detentora e Solicitantes preenchidos 
df_contratos_int = df_contratos_int[
    (~df_contratos_int['PROCESSO_ANATEL'].isna())  & 
    (~df_contratos_int['PRESTADORA_1'].isna())  & 
    (~df_contratos_int['PRESTADORA_2'].isna()) 
]
df_contratos_int

,PROCESSO_ANATEL,PROTOCOLO_DATA,ACORDO_TIPO,PRESTADORA_1_CNPJ,PRESTADORA_1,SERVICO_1,MODALIDADE_STFC_1,PRESTADORA_2_CNPJ,PRESTADORA_2,SERVICO_2,MODALIDADE_STFC_2,DESPACHO_DECISORIO_SEI,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA,OBSERVACAO
0,53500.028940/2006-92,2006-10-26,NÃO INFORMADO,NaN,VIVO S.A.,10,INDEFINIDA,66.970.229/0001-67,CLARO NXT TELECOMUNICACOES LTDA,20,INDEFINIDA,-1,2014-10-06,2014-10-06,Classe: IV. Localização: ARQUIVO GERAL.
1,53500.004342/2007-17,2007-02-14,CONTRATO,NaN,VIVO S.A.,10,INDEFINIDA,66.970.229/0001-67,CLARO NXT TELECOMUNICACOES LTDA,20,INDEFINIDA,47,2015-08-05,2015-08-19,Classe: IV. Localização: ESTAÇÃO.
2,53500.021964/2008-82,2008-08-14,NÃO INFORMADO,66.970.229/0001-67,CLARO NXT TELECOMUNICACOES LTDA,10,INDEFINIDA,NaN,UNICEL DO BRASIL TELECOMUNICAÇÕES LTDA.,10,INDEFINIDA,3,2013-06-13,2013-06-17,Classe: IV. Localização: ARQUIVO SETORIAL.
3,53500.004720/2009-16,2009-03-04,NÃO INFORMADO,NaN,VIVO S.A.,10,INDEFINIDA,NaN,UNICEL DO BRASIL TELECOMUNICAÇÕES LTDA.,10,INDEFINIDA,2,2013-06-13,2013-06-17,Classe: IV. Localização: ARQUIVO SETORIAL.
4,53500.007610/2009-14,2009-03-19,NÃO INFORMADO,66.970.229/0001-67,CLARO NXT TELECOMUNICACOES LTDA,20,INDEFINIDA,NaN,AGERA TELECOMUNICAÇÕES S.A.,20,INDEFINIDA,4998,2014-09-26,2014-09-29,Classe: IV. Localização: ESTAÇÃO.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12369,53500.019526/2025-55,2025-03-17,CONTRATO,NaN,TIM S A,171,LDN,28.135.347/0001-60,4NET SOLUCOES EM TELECOMUNICACOES LTDA,171,LDN,13578256,2025-04-17,2025-04-17,Homologado por ORPA/OPI
12370,53500.019526/2025-55,2025-03-17,CONTRATO,NaN,TIM S A,171,LDN,28.135.347/0001-60,4NET SOLUCOES EM TELECOMUNICACOES LTDA,171,LOCAL,13578256,2025-04-17,2025-04-17,Homologado por ORPA/OPI
12371,53500.019526/2025-55,2025-03-17,CONTRATO,NaN,TIM S A,171,LOCAL,28.135.347/0001-60,4NET SOLUCOES EM TELECOMUNICACOES LTDA,171,LDI,13578256,2025-04-17,2025-04-17,Homologado por ORPA/OPI
12372,53500.019526/2025-55,2025-03-17,CONTRATO,NaN,TIM S A,171,LOCAL,28.135.347/0001-60,4NET SOLUCOES EM TELECOMUNICACOES LTDA,171,LDN,13578256,2025-04-17,2025-04-17,Homologado por ORPA/OPI


#### Remoção de Duplicadas (novamente )

In [18]:
print(f'Tamanho dataset ANTES: {df_contratos_int.shape}')
df_contratos_int = df_contratos_int.drop_duplicates()
print(f'Tamanho dataset DEPOIS: {df_contratos_int.shape}')

Tamanho dataset ANTES: (12364, 15)
Tamanho dataset DEPOIS: (12364, 15)


#### Coluna de versionamento 

Vammos criar uma coluna para salvar as informações de versionamento de um contrato: 

* `Num Sequencia`, qual é a versão que aquele contrato está
* `final`, se aquela é a versão final ou não. 



In [78]:
df_contratos_int = (
    df_contratos_int
        .sort_values(by=['PROCESSO_ANATEL', 'PROTOCOLO_DATA'], kind='stable')
)
df_contratos_int['NUM_SEQUENCIA'] = df_contratos_int.groupby(by='PROCESSO_ANATEL').cumcount() + 1
df_max_seq = (
    df_contratos_int
        .groupby('PROCESSO_ANATEL')
        [['NUM_SEQUENCIA']]
        .max()
        .reset_index()
        .rename(columns={'NUM_SEQUENCIA': 'MAX_NUM_SEQ'})
)
df_contratos_int = df_contratos_int.merge(df_max_seq, on='PROCESSO_ANATEL', how='left')
df_contratos_int['FINAL']  = df_contratos_int['NUM_SEQUENCIA'] == df_contratos_int['MAX_NUM_SEQ']
df_contratos_int = df_contratos_int.drop(columns=['MAX_NUM_SEQ'])
df_contratos_int

,PROCESSO_ANATEL,PROTOCOLO_DATA,ACORDO_TIPO,PRESTADORA_1_CNPJ,PRESTADORA_1,SERVICO_1,MODALIDADE_STFC_1,PRESTADORA_2_CNPJ,PRESTADORA_2,SERVICO_2,MODALIDADE_STFC_2,DESPACHO_DECISORIO_SEI,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA,OBSERVACAO,NUM_SEQUENCIA,FINAL
0,53500.000052/2006-13,2014-10-20,ADITIVO,NaN,TIM CELULAR S.A.,10,INDEFINIDA,76.535.764/0001-43,OI S.A. - EM RECUPERACAO JUDICIAL,171,LDI,1,2014-12-26,2014-12-31,Classe: II.,1,False
1,53500.000052/2006-13,2014-10-20,ADITIVO,NaN,TIM CELULAR S.A.,10,INDEFINIDA,76.535.764/0001-43,OI S.A. - EM RECUPERACAO JUDICIAL,171,LDN,1,2014-12-26,2014-12-31,Classe: II.,2,True
2,53500.000081/2020-25,2021-05-04,CONTRATO,NaN,TIM S A,10,INDEFINIDA,NaN,DEFFERRARI SOLUCOES EM INTERNET LTDA,171,LDI,7176936,2021-07-26,2021-07-26,Homologado por ORPA/OPI.,1,False
3,53500.000081/2020-25,2021-05-04,CONTRATO,NaN,TIM S A,10,INDEFINIDA,NaN,DEFFERRARI SOLUCOES EM INTERNET LTDA,171,LDN,7176936,2021-07-26,2021-07-26,Homologado por ORPA/OPI.,2,False
4,53500.000081/2020-25,2021-05-04,CONTRATO,NaN,TIM S A,10,INDEFINIDA,NaN,DEFFERRARI SOLUCOES EM INTERNET LTDA,171,LOCAL,7176936,2021-07-26,2021-07-26,Homologado por ORPA/OPI.,3,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12359,53516.006349/2015-04,2015-10-27,CONTRATO,NaN,GLOBAL VILLAGE TELECOM LTDA,171,LOCAL,NaN,COMPANHIA ITABIRANA DE TELECOMUNICACOES LTDA,171,LDI,29,2016-03-18,2016-03-24,Classe: I. Localização: SEI.,7,False
12360,53516.006349/2015-04,2015-10-27,CONTRATO,NaN,GLOBAL VILLAGE TELECOM LTDA,171,LOCAL,NaN,COMPANHIA ITABIRANA DE TELECOMUNICACOES LTDA,171,LDN,29,2016-03-18,2016-03-24,Classe: I. Localização: SEI.,8,False
12361,53516.006349/2015-04,2015-10-27,CONTRATO,NaN,GLOBAL VILLAGE TELECOM LTDA,171,LOCAL,NaN,COMPANHIA ITABIRANA DE TELECOMUNICACOES LTDA,171,LOCAL,29,2016-03-18,2016-03-24,Classe: I. Localização: SEI.,9,True
12362,53528.001854/2017-87,2017-05-19,ADITIVO,NaN,TIM CELULAR S.A.,10,INDEFINIDA,13.114.336/0001-27,PORTO VELHO TELECOMUNICAÇÕES LTDA,171,LOCAL,1624188,2017-07-07,2017-07-11,Classe: II.,1,True


### Contratos - MVNO 

In [19]:
df_contratos_mvno = pd.read_csv(
    r'Dados - Contratos/contratos_mvno.csv',
    sep = ';'
)

In [20]:
df_contratos_mvno.columns

Index(['ACORDO_TIPO', 'PROTOCOLO_DATA', 'PROCESSO_ANATEL', 'PRESTADORA_ORIGEM',
       'PRESTADORA_ORIGEM_CNPJ', 'CREDENCIADA', 'CREDENCIADA_CNPJ',
       'VIGENCIA_DATA_FIM', 'PROCESSO_DESCREDENCIAMENTO', 'OBSERVACAO',
       'DESPACHO_DECISORIO_SEI', 'DESPACHO_DECISORIO_DATA', 'CONCLUSAO_DATA'],
      dtype='str')

#### Tratamento de Duplicas 

In [21]:
print(f'Tamanho dataset ANTES: {df_contratos_mvno.shape}')
df_contratos_mvno = df_contratos_mvno.drop_duplicates()
print(f'Tamanho dataset DEPOIS: {df_contratos_mvno.shape}')

Tamanho dataset ANTES: (341, 13)
Tamanho dataset DEPOIS: (341, 13)


####  Tratamento Colunas de Data 

In [22]:
# Vamos colocar como nulo, quando a data de fim de vigencia for uma string = vigencia
df_contratos_mvno.loc[df_contratos_mvno.VIGENCIA_DATA_FIM == 'VIGENTE', 'VIGENCIA_DATA_FIM'] = np.nan

In [23]:
colunas_data = ['PROTOCOLO_DATA','DESPACHO_DECISORIO_DATA', 'CONCLUSAO_DATA', 'VIGENCIA_DATA_FIM']
df_contratos_mvno[colunas_data].isna().sum()
for col in colunas_data:
    df_contratos_mvno[col] = pd.to_datetime(
        df_contratos_mvno[col],
        format='%d/%m/%Y',
        errors='coerce'
    )
df_contratos_mvno[colunas_data].sample(5)

,PROTOCOLO_DATA,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA,VIGENCIA_DATA_FIM
48,2020-07-02,2020-07-30,2020-07-30,2024-10-02
309,2025-01-17,2025-02-27,2025-03-10,NaT
12,2017-09-04,2018-05-14,2018-05-14,NaT
57,2020-07-29,2020-08-05,2020-08-05,NaT
316,2025-02-28,2025-03-17,2025-03-18,NaT


#### Tratamento de Colunas de TEXTO 

In [24]:
colunas_texto = df_contratos_mvno.select_dtypes(include=['object', 'string']).columns
colunas_texto

Index(['ACORDO_TIPO', 'PROCESSO_ANATEL', 'PRESTADORA_ORIGEM', 'CREDENCIADA',
       'PROCESSO_DESCREDENCIAMENTO', 'OBSERVACAO'],
      dtype='str')

In [25]:
df_contratos_mvno.head(5)

,ACORDO_TIPO,PROTOCOLO_DATA,PROCESSO_ANATEL,PRESTADORA_ORIGEM,PRESTADORA_ORIGEM_CNPJ,CREDENCIADA,CREDENCIADA_CNPJ,VIGENCIA_DATA_FIM,PROCESSO_DESCREDENCIAMENTO,OBSERVACAO,DESPACHO_DECISORIO_SEI,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA
0,CONTRATO,2015-01-07,53500.000262/2015-94,Telefonica Brasil S.a.,2558157000162,Mais AD Credenciada de Telefonia S.A.,17751901000118,2020-04-07,53500.000262/2015-94,Publicação no DOU.,201590015890,2015-01-30,2015-01-30
1,CONTRATO,2016-03-14,53504.001925/2016-39,Datora Mobile Telecomunicacoes S.a.,18384930000151,BT Brasil Serviços de Telecomunicações Ltda. (...,33179565000137,NaT,NaN,Não identificado,588202,2016-07-20,2016-07-20
2,CONTRATO,2016-03-23,53500.004886/2016-61,SURF TELECOM SA,10455746000143,Always Tecnologia Ltda. (Igreja da Fé),69190577000100,2017-07-21,NaN,Não identificado,591183,2016-07-20,2016-07-20
3,CONTRATO,2016-03-23,53500.006148/2016-59,SURF TELECOM SA,10455746000143,Lanis Redes e Consultoria Ltda. (Lanis),9526734000183,2019-01-25,53500.006148/2016-59,De acordo com a coorespondência SEI nº 3754922.,680889,2016-08-24,2016-08-24
4,CONTRATO,2016-06-24,53500.015283/2016-95,SURF TELECOM SA,10455746000143,Empresa Brasileira de Correios e Telégrafos (C...,34028316000103,NaT,NaN,Não identificado,930841,2016-11-17,2016-11-17


In [26]:
# Normalizando padrão textual do ACOROD_TIPO e preenchendo com "INDEFINIDO" para nulos
df_contratos_mvno['ACORDO_TIPO'] = (
    df_contratos_mvno['ACORDO_TIPO']
    .apply(normalizar_texto)
    .str.upper()
    .fillna('INDEFINIDO')
)

# Filtrando apenas registros de processos válidos 
df_contratos_mvno['PROCESSO_ANATEL'] = (
    df_contratos_mvno['PROCESSO_ANATEL']
        .apply(normalizar_texto)
        .apply(filtrar_processos_validos)
)

# limpando dados de processos anatel de descredenciamento, mantendo apenas registros válidos
df_contratos_mvno['PROCESSO_DESCREDENCIAMENTO'] = (
    df_contratos_mvno['PROCESSO_DESCREDENCIAMENTO']
        .apply(normalizar_texto)
        .apply(filtrar_processos_validos)
)

# Normalizando padrão textual do PRESTADOR Origem e CREDENCIADA; preenchendo com "INDEFINIDO" para nulos
df_contratos_mvno['PRESTADORA_ORIGEM'] = (
    df_contratos_mvno['PRESTADORA_ORIGEM']
    .apply(normalizar_texto)
    .str.upper()
)
df_contratos_mvno['CREDENCIADA'] = (
    df_contratos_mvno['CREDENCIADA']
    .apply(normalizar_texto)
    .str.upper()
)

# Normalizando os CNPJS da PRESTADORA ORIGEM e da CREDENCIADA. As vezes não teremos CNPJ preenchido, mas temos que ter
# ao menos o nome da empresa 

df_contratos_mvno['PRESTADORA_ORIGEM_CNPJ'] = (
    df_contratos_mvno['PRESTADORA_ORIGEM_CNPJ']
        .apply(normalizar_texto)
        .str.replace(' ', '')
        .apply(extract_valid_cnpjs)
        .apply(format_cnpj)
    )

df_contratos_mvno['CREDENCIADA_CNPJ'] = (
    df_contratos_mvno['CREDENCIADA_CNPJ']
        .apply(normalizar_texto)
        .str.replace(' ', '')
        .apply(extract_valid_cnpjs)
        .apply(format_cnpj)
    )

# Limpando DESPACHO DECISORIO SEI, um valor INTEIRO mas as vezes vem com texto sujo ou nulo.
df_contratos_mvno['DESPACHO_DECISORIO_SEI'] = (
    df_contratos_mvno['DESPACHO_DECISORIO_SEI']
    .apply(normalizar_texto)
    .apply(limpeza_sei)
)

# O campo de Observações é uma área de texto llivre, vamos só tirar valores problematicos e manter null quando
# necessário 
df_contratos_mvno['OBSERVACAO'] = (
    df_contratos_mvno['OBSERVACAO']
        .str.strip()
        .map(lambda x: np.nan if pd.isna(x) or len(str(x)) == 0 else str(x))
        .astype(str)        
    )


# # Selecionando apenas as linhas com processo anatel, Detentora e Solicitantes preenchidos 
df_contratos_mvno = df_contratos_mvno[
    (~df_contratos_mvno['PROCESSO_ANATEL'].isna())  & 
    (~df_contratos_mvno['PRESTADORA_ORIGEM'].isna())  & 
    (~df_contratos_mvno['CREDENCIADA'].isna()) 
]
df_contratos_mvno

,ACORDO_TIPO,PROTOCOLO_DATA,PROCESSO_ANATEL,PRESTADORA_ORIGEM,PRESTADORA_ORIGEM_CNPJ,CREDENCIADA,CREDENCIADA_CNPJ,VIGENCIA_DATA_FIM,PROCESSO_DESCREDENCIAMENTO,OBSERVACAO,DESPACHO_DECISORIO_SEI,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA
0,CONTRATO,2015-01-07,53500.000262/2015-94,TELEFONICA BRASIL S.A.,NaN,MAIS AD CREDENCIADA DE TELEFONIA S.A.,17.751.901/0001-18,2020-04-07,53500.000262/2015-94,Publicação no DOU.,201590015890,2015-01-30,2015-01-30
1,CONTRATO,2016-03-14,53504.001925/2016-39,DATORA MOBILE TELECOMUNICACOES S.A.,18.384.930/0001-51,BT BRASIL SERVIÇOS DE TELECOMUNICAÇÕES LTDA. (...,33.179.565/0001-37,NaT,NaN,Não identificado,588202,2016-07-20,2016-07-20
2,CONTRATO,2016-03-23,53500.004886/2016-61,SURF TELECOM SA,10.455.746/0001-43,ALWAYS TECNOLOGIA LTDA. (IGREJA DA FÉ),69.190.577/0001-00,2017-07-21,NaN,Não identificado,591183,2016-07-20,2016-07-20
3,CONTRATO,2016-03-23,53500.006148/2016-59,SURF TELECOM SA,10.455.746/0001-43,LANIS REDES E CONSULTORIA LTDA. (LANIS),NaN,2019-01-25,53500.006148/2016-59,De acordo com a coorespondência SEI nº 3754922.,680889,2016-08-24,2016-08-24
4,CONTRATO,2016-06-24,53500.015283/2016-95,SURF TELECOM SA,10.455.746/0001-43,EMPRESA BRASILEIRA DE CORREIOS E TELÉGRAFOS (C...,34.028.316/0001-03,NaT,NaN,Não identificado,930841,2016-11-17,2016-11-17
...,...,...,...,...,...,...,...,...,...,...,...,...,...
336,CONTRATO,2025-05-21,53500.037709/2025-52,SURF TELECOM SA,10.455.746/0001-43,BR SISTEMAS LTDA,49.388.765/0001-30,NaT,NaN,Não identificado,13759227,2025-05-29,2025-06-02
337,CONTRATO,2025-05-21,53500.037707/2025-63,SURF TELECOM SA,10.455.746/0001-43,BCS11 PARTICIPAÇÕES S.A.,39.414.947/0001-84,NaT,NaN,Não identificado,13750816,2025-05-29,2025-06-02
338,CONTRATO,2025-05-23,53500.038563/2025-62,SURF TELECOM SA,10.455.746/0001-43,LIVE OPTICAL FIBER LTDA,48.226.378/0001-34,NaT,NaN,Não identificado,13827794,2025-06-12,2025-06-16
339,ADITIVO,2025-05-29,53500.025367/2020-13,SURF TELECOM SA,10.455.746/0001-43,PLAY MÓVEL LTDA,44.255.627/0001-69,NaT,NaN,17º Aditivo,13834491,2025-06-12,2025-06-16


#### Remoção de Duplicadas (novamente )

In [27]:
print(f'Tamanho dataset ANTES: {df_contratos_mvno.shape}')
df_contratos_mvno = df_contratos_mvno.drop_duplicates()
print(f'Tamanho dataset DEPOIS: {df_contratos_mvno.shape}')

Tamanho dataset ANTES: (341, 13)
Tamanho dataset DEPOIS: (341, 13)


#### Coluna de versionamento 

Vammos criar uma coluna para salvar as informações de versionamento de um contrato: 

* `Num Sequencia`, qual é a versão que aquele contrato está
* `final`, se aquela é a versão final ou não. 



In [81]:
df_contratos_mvno = (
    df_contratos_mvno
        .sort_values(by=['PROCESSO_ANATEL', 'PROTOCOLO_DATA'], kind='stable')
)
df_contratos_mvno['NUM_SEQUENCIA'] = df_contratos_mvno.groupby(by='PROCESSO_ANATEL').cumcount() + 1
df_max_seq = (
    df_contratos_mvno
        .groupby('PROCESSO_ANATEL')
        [['NUM_SEQUENCIA']]
        .max()
        .reset_index()
        .rename(columns={'NUM_SEQUENCIA': 'MAX_NUM_SEQ'})
)
df_contratos_mvno = df_contratos_mvno.merge(df_max_seq, on='PROCESSO_ANATEL', how='left')
df_contratos_mvno['FINAL']  = df_contratos_mvno['NUM_SEQUENCIA'] == df_contratos_mvno['MAX_NUM_SEQ']
df_contratos_mvno = df_contratos_mvno.drop(columns=['MAX_NUM_SEQ'])
df_contratos_mvno

,ACORDO_TIPO,PROTOCOLO_DATA,PROCESSO_ANATEL,PRESTADORA_ORIGEM,PRESTADORA_ORIGEM_CNPJ,CREDENCIADA,CREDENCIADA_CNPJ,VIGENCIA_DATA_FIM,PROCESSO_DESCREDENCIAMENTO,OBSERVACAO,DESPACHO_DECISORIO_SEI,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA,NUM_SEQUENCIA,FINAL
0,CONTRATO,2015-01-07,53500.000262/2015-94,TELEFONICA BRASIL S.A.,NaN,MAIS AD CREDENCIADA DE TELEFONIA S.A.,17.751.901/0001-18,2020-04-07,53500.000262/2015-94,Publicação no DOU.,201590015890,2015-01-30,2015-01-30,1,False
1,ADITIVO,2017-05-23,53500.000262/2015-94,TELEFONICA BRASIL S.A.,NaN,MAIS AD CREDENCIADA DE TELEFONIA S.A.,17.751.901/0001-18,2020-04-07,53500.000262/2015-94,1º Aditivo,1927446,2017-11-09,2017-11-09,2,True
2,CONTRATO,2022-01-04,53500.000563/2022-47,TELEXPERTS TELECOMUNICACOES S.A.,NaN,P4 TELECOM LTDA,10.703.677/0001-40,NaT,NaN,Não identificado,7940722,2022-01-24,2022-01-24,1,True
3,CONTRATO,2019-12-02,53500.001132/2020-36,SURF TELECOM SA,10.455.746/0001-43,WANTEL TECNOLOGIA LTDA - PPP,21.850.223/0001-18,2023-03-22,53500.001132/2020-36,De acordo com comunicado SEI nº 10028140. Proc...,5243988,2020-02-21,2020-02-21,1,True
4,CONTRATO,2023-01-10,53500.001813/2023-47,NEXT LEVEL TELECOM LTDA.,20.877.748/0001-84,KORE BRASIL LTDA,24.492.478/0001-44,NaT,NaN,Não identificado,9681125,2023-01-16,2023-01-16,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336,ADITIVO,2020-10-29,53504.008572/2017-89,TELEFONICA BRASIL S.A.,NaN,BRISANET SERVIÇOS DE TELECOMUNICAÇÕES LTDA,NaN,NaT,NaN,2º Aditivo,6226032,2020-11-27,2020-11-27,3,False
337,ADITIVO,2021-04-07,53504.008572/2017-89,TELEFONICA BRASIL S.A.,NaN,BRISANET SERVIÇOS DE TELECOMUNICAÇÕES LTDA,NaN,NaT,NaN,3º Aditivo,6885419,2021-05-19,2021-05-19,4,False
338,ADITIVO,2021-11-17,53504.008572/2017-89,TELEFONICA BRASIL S.A.,NaN,BRISANET SERVIÇOS DE TELECOMUNICAÇÕES LTDA,NaN,NaT,NaN,4º Aditivo,7766123,2021-12-13,2021-12-13,5,True
339,CONTRATO,2017-11-22,53504.016428/2017-16,DATORA MOBILE TELECOMUNICACOES S.A.,18.384.930/0001-51,FONELIGHT TELECOMUNICAÇÕES S/A,NaN,NaT,NaN,Não identificado,2263762,2018-01-26,2018-01-26,1,True


### Contratos - Ran Sharing 

In [28]:
df_contratos_ran_sharing = pd.read_csv(
    r'Dados - Contratos/contratos_ran_sharing.csv',
    sep=';'
)
print(df_contratos_ran_sharing.shape)
df_contratos_ran_sharing.head()

(17, 20)


,PROCESSO_ANATEL,PROTOCOLO_DATA,ACORDO_TIPO,PROCESSO_ORIGEM,PRESTADORA_1_CNPJ,PRESTADORA_1,PRESTADORA_2_CNPJ,PRESTADORA_2,PRESTADORA_3_CNPJ,PRESTADORA_3,PRESTADORA_4_CNPJ,PRESTADORA_4,ACORDO_TECNOLOGIA,INFORME,INFORME_SEI,INFORME_DATA,ACORDAO,ACORDAO_SEI,ACORDAO_DATA,OBSERVACAO
0,53500.031688/2012-47,14/11/2012,CONTRATO,NaN,4206050000180,TIM CELULAR S.A.,5423963000111,OI MÓVEL S.A. - EM RECUPERAÇÃO JUDICIAL,NaN,NaN,NaN,NaN,MORAN,Informe nº 01/2013/PVCPR/PVCP/SPV,2754301,21/01/2013,Despacho nº 2719/2013-CD,2754301,21/01/2013,Processo iniciado no SICAP.\r
1,53500.001341/2014-31,09/01/2014,CONTRATO,NaN,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,2558157000162,Telefonica Brasil S.a.,NaN,NaN,NaN,NaN,MOCN,Informe nº 111/2014-PRRE/CPRP/SPR/SCP,2841116,18/02/2014,Acórdão nº 133/2014,2841116,18/02/2014,Processo iniciado no SICAP.\r
2,53500.017260/2015-34,29/07/2015,CONTRATO,NaN,4206050000180,TIM CELULAR S.A.,2558157000162,Telefonica Brasil S.a.,5.423963e+12,OI MÓVEL S.A. - EM RECUPERAÇÃO JUDICIAL,NaN,NaN,MOCN,Informe nº 288/2015-PRRE/ORLE/ORER/CPRP/SPR/SO...,508651,08/10/2015,Acórdão nº 555/2015,508651,08/10/2015,Processo iniciado no SICAP.\r
3,53500.001089/2014-61,14/02/2014,CONTRATO,NaN,40432544000147,CLARO S.A.,2558157000162,Telefonica Brasil S.a.,NaN,NaN,NaN,NaN,MOCN,Informe nº 189/2014-PRRE/CPRP/SPR/SCP,392130,03/04/2014,Acórdão nº 199/2014,392130,03/04/2014,Processo iniciado no SICAP.\r
4,53500.020308/2014-19,05/09/2014,ADITIVO,53500.001089/2014-61,40432544000147,CLARO S.A.,2558157000162,Telefonica Brasil S.a.,NaN,NaN,NaN,NaN,MOCN,Informe nº 733/2014-CPRP/PRRE/SCP/SPR,392142,05/12/2014,Acórdão nº 22/2015,392142,05/12/2014,Processo iniciado no SICAP.\r


In [29]:
df_contratos_ran_sharing.columns

Index(['PROCESSO_ANATEL', 'PROTOCOLO_DATA', 'ACORDO_TIPO', 'PROCESSO_ORIGEM',
       'PRESTADORA_1_CNPJ', 'PRESTADORA_1', 'PRESTADORA_2_CNPJ',
       'PRESTADORA_2', 'PRESTADORA_3_CNPJ', 'PRESTADORA_3',
       'PRESTADORA_4_CNPJ', 'PRESTADORA_4', 'ACORDO_TECNOLOGIA', 'INFORME',
       'INFORME_SEI', 'INFORME_DATA', 'ACORDAO', 'ACORDAO_SEI', 'ACORDAO_DATA',
       'OBSERVACAO'],
      dtype='str')

#### Tratamento de Duplicas 

In [30]:
print(f'Tamanho dataset ANTES: {df_contratos_ran_sharing.shape}')
df_contratos_ran_sharing = df_contratos_ran_sharing.drop_duplicates()
print(f'Tamanho dataset DEPOIS: {df_contratos_ran_sharing.shape}')

Tamanho dataset ANTES: (17, 20)
Tamanho dataset DEPOIS: (16, 20)


####  Tratamento Colunas de Data 

In [31]:
colunas_data = ['PROTOCOLO_DATA','INFORME_DATA','ACORDAO_DATA']
df_contratos_ran_sharing[colunas_data].isna().sum()
for col in colunas_data:
    df_contratos_ran_sharing[col] = pd.to_datetime(
        df_contratos_ran_sharing[col],
        format='%d/%m/%Y',
        errors='coerce'
    )
df_contratos_ran_sharing[colunas_data].sample(5)

,PROTOCOLO_DATA,INFORME_DATA,ACORDAO_DATA
10,2018-09-20,2019-03-22,2019-03-22
16,2022-09-16,2023-05-09,2023-05-09
14,2021-03-15,2021-04-09,2021-04-09
11,2019-12-26,2020-02-07,2020-02-07
2,2015-07-29,2015-10-08,2015-10-08


#### Removendo Colunas Nulas 

In [32]:
print(df_contratos_ran_sharing.PRESTADORA_4_CNPJ.isna().value_counts())
print(df_contratos_ran_sharing.PRESTADORA_4.isna().value_counts())

PRESTADORA_4_CNPJ
True    16
Name: count, dtype: int64
PRESTADORA_4
True    16
Name: count, dtype: int64


In [33]:
df_contratos_ran_sharing = df_contratos_ran_sharing.drop(columns=['PRESTADORA_4_CNPJ','PRESTADORA_4'])
df_contratos_ran_sharing

,PROCESSO_ANATEL,PROTOCOLO_DATA,ACORDO_TIPO,PROCESSO_ORIGEM,PRESTADORA_1_CNPJ,PRESTADORA_1,PRESTADORA_2_CNPJ,PRESTADORA_2,PRESTADORA_3_CNPJ,PRESTADORA_3,ACORDO_TECNOLOGIA,INFORME,INFORME_SEI,INFORME_DATA,ACORDAO,ACORDAO_SEI,ACORDAO_DATA,OBSERVACAO
0,53500.031688/2012-47,2012-11-14,CONTRATO,NaN,4206050000180,TIM CELULAR S.A.,5423963000111,OI MÓVEL S.A. - EM RECUPERAÇÃO JUDICIAL,NaN,NaN,MORAN,Informe nº 01/2013/PVCPR/PVCP/SPV,2754301,2013-01-21,Despacho nº 2719/2013-CD,2754301,2013-01-21,Processo iniciado no SICAP.\r
1,53500.001341/2014-31,2014-01-09,CONTRATO,NaN,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,2558157000162,Telefonica Brasil S.a.,NaN,NaN,MOCN,Informe nº 111/2014-PRRE/CPRP/SPR/SCP,2841116,2014-02-18,Acórdão nº 133/2014,2841116,2014-02-18,Processo iniciado no SICAP.\r
2,53500.017260/2015-34,2015-07-29,CONTRATO,NaN,4206050000180,TIM CELULAR S.A.,2558157000162,Telefonica Brasil S.a.,5.423963e+12,OI MÓVEL S.A. - EM RECUPERAÇÃO JUDICIAL,MOCN,Informe nº 288/2015-PRRE/ORLE/ORER/CPRP/SPR/SO...,508651,2015-10-08,Acórdão nº 555/2015,508651,2015-10-08,Processo iniciado no SICAP.\r
3,53500.001089/2014-61,2014-02-14,CONTRATO,NaN,40432544000147,CLARO S.A.,2558157000162,Telefonica Brasil S.a.,NaN,NaN,MOCN,Informe nº 189/2014-PRRE/CPRP/SPR/SCP,392130,2014-04-03,Acórdão nº 199/2014,392130,2014-04-03,Processo iniciado no SICAP.\r
4,53500.020308/2014-19,2014-09-05,ADITIVO,53500.001089/2014-61,40432544000147,CLARO S.A.,2558157000162,Telefonica Brasil S.a.,NaN,NaN,MOCN,Informe nº 733/2014-CPRP/PRRE/SCP/SPR,392142,2014-12-05,Acórdão nº 22/2015,392142,2014-12-05,Processo iniciado no SICAP.\r
5,53500.024608/2014-69,2014-10-22,ADITIVO,53500.001089/2014-61,40432544000147,CLARO S.A.,2558157000162,Telefonica Brasil S.a.,NaN,NaN,MOCN,Informe nº 733/2014-CPRP/PRRE/SCP/SPR,392142,2014-12-05,Acórdão nº 22/2015,392142,2014-12-05,Processo iniciado no SICAP.\r
6,53500.002241/2016-94,2016-02-02,ADITIVO,53500.001089/2014-61,40432544000147,CLARO S.A.,2558157000162,Telefonica Brasil S.a.,NaN,NaN,MOCN,Informe nº 83/2016/SEI/CPRP/SCP,345635,2016-04-05,Acórdão nº 194/2016,526288,2016-04-05,\r
7,53500.002243/2016-83,2016-02-02,ADITIVO,53500.001089/2014-61,40432544000147,CLARO S.A.,2558157000162,Telefonica Brasil S.a.,NaN,NaN,MOCN,Informe nº 58/2016/SEI/CPRP/SCP,302087,2016-04-05,Acórdão nº 194/2016,526288,2016-04-05,\r
8,53500.010657/2016-86,2016-05-09,CONTRATO,NaN,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,2558157000162,Telefonica Brasil S.a.,NaN,NaN,GWCN,Informe nº 179/2016/SEI/CPRP/SCP,504309,2016-06-28,Acórdão nº 281/2016,720364,2016-06-28,\r
9,53500.011812/2018-43,2018-03-29,ADITIVO,53500.031688/2012-47,4206050000180,TIM CELULAR S.A.,5423963000111,OI MÓVEL S.A. - EM RECUPERAÇÃO JUDICIAL,7.653576e+13,Oi S.a. - em Recuperacao Judicial,MOCN,Informe nº 180/2018/SEI/CPRP/SCP,2861810,2018-06-21,Acórdão nº 409/2018,2991702,2018-06-21,\r


#### Tratamento de Colunas de TEXTO 

In [34]:
colunas_texto = df_contratos_ran_sharing.select_dtypes(include=['object', 'string']).columns
colunas_texto

Index(['PROCESSO_ANATEL', 'ACORDO_TIPO', 'PROCESSO_ORIGEM', 'PRESTADORA_1',
       'PRESTADORA_2', 'PRESTADORA_3', 'ACORDO_TECNOLOGIA', 'INFORME',
       'ACORDAO', 'OBSERVACAO'],
      dtype='str')

In [35]:
# Normalizando padrão textual do ACOROD_TIPO e preenchendo com "INDEFINIDO" para nulos
df_contratos_ran_sharing['ACORDO_TIPO'] = (
    df_contratos_ran_sharing['ACORDO_TIPO']
    .apply(normalizar_texto)
    .str.upper()
    .fillna('INDEFINIDO')
)

# Filtrando apenas registros de processos válidos 
df_contratos_ran_sharing['PROCESSO_ANATEL'] = (
    df_contratos_ran_sharing['PROCESSO_ANATEL']
        .apply(normalizar_texto)
        .apply(filtrar_processos_validos)
)

# limpando dados de processos anatel de origim, mantendo apenas registros válidos

df_contratos_ran_sharing['PROCESSO_ORIGEM'] = (
    df_contratos_ran_sharing['PROCESSO_ORIGEM']
        .apply(normalizar_texto)
        .apply(filtrar_processos_validos)
)


# # Normalizando padrão textual do PRESTADORES preenchendo com "INDEFINIDO" para nulos
for prestadores in ('PRESTADORA_1', 'PRESTADORA_2', 'PRESTADORA_3'):
    df_contratos_ran_sharing[prestadores] = (
        df_contratos_ran_sharing[prestadores]
        .apply(normalizar_texto)
        .str.upper()
    )

# # Normalizando os CNPJS dos prestradores. As vezes não teremos CNPJ preenchido, mas temos que ter
# # ao menos o nome da empresa 
for prestadores in ('PRESTADORA_1_CNPJ', 'PRESTADORA_2_CNPJ', 'PRESTADORA_3_CNPJ'):
    df_contratos_ran_sharing[prestadores] = (
        df_contratos_ran_sharing[prestadores]
        .apply(normalizar_texto)
        .str.replace(' ', '')
        .apply(extract_valid_cnpjs)
        .apply(format_cnpj)
    )

# O Campod e acordo tecnologia tem um padrão já bem definido, mas só por garantia vamos aplicar uma normalização
# simples para textos
df_contratos_ran_sharing['ACORDO_TECNOLOGIA'] = (
    df_contratos_ran_sharing['ACORDO_TECNOLOGIA']
    .apply(normalizar_texto)
    .str.upper()
    .fillna('INDEFINIDO')
    .astype(str)        
)


# Tratamento Básico sobre a coluna de Acordão
df_contratos_ran_sharing['ACORDAO'] = (
    df_contratos_ran_sharing['ACORDAO']
    .apply(normalizar_texto)
    .str.upper()
    .fillna('INDEFINIDO')
    .astype(str)        
)


# O campo de Observações é uma área de texto llivre, vamos só tirar valores problematicos e manter null quando
# necessário 
df_contratos_ran_sharing['OBSERVACAO'] = (
    df_contratos_ran_sharing['OBSERVACAO']
        .str.strip()
        .map(lambda x: np.nan if pd.isna(x) or len(str(x)) == 0 else str(x))
        .astype(str)        
    )


# Tratamento Básico sobre a coluna de Acordão
df_contratos_ran_sharing['INFORME'] = (
    df_contratos_ran_sharing['INFORME']
        .str.strip()
        .map(lambda x: np.nan if pd.isna(x) or len(str(x)) == 0 else str(x))
        .astype(str)        
    )


# # Selecionando apenas as linhas com processo anatel, Detentora e Solicitantes preenchidos 
df_contratos_ran_sharing = df_contratos_ran_sharing[
    (~df_contratos_ran_sharing['PROCESSO_ANATEL'].isna())  & 
    (~df_contratos_ran_sharing['PRESTADORA_1'].isna()) & 
    (~df_contratos_ran_sharing['ACORDAO'].isna())
]
df_contratos_ran_sharing

,PROCESSO_ANATEL,PROTOCOLO_DATA,ACORDO_TIPO,PROCESSO_ORIGEM,PRESTADORA_1_CNPJ,PRESTADORA_1,PRESTADORA_2_CNPJ,PRESTADORA_2,PRESTADORA_3_CNPJ,PRESTADORA_3,ACORDO_TECNOLOGIA,INFORME,INFORME_SEI,INFORME_DATA,ACORDAO,ACORDAO_SEI,ACORDAO_DATA,OBSERVACAO
0,53500.031688/2012-47,2012-11-14,CONTRATO,NaN,NaN,TIM CELULAR S.A.,NaN,OI MÓVEL S.A. - EM RECUPERAÇÃO JUDICIAL,NaN,NaN,MORAN,Informe nº 01/2013/PVCPR/PVCP/SPV,2754301,2013-01-21,DESPACHO Nº 2719/2013-CD,2754301,2013-01-21,Processo iniciado no SICAP.
1,53500.001341/2014-31,2014-01-09,CONTRATO,NaN,66.970.229/0001-67,CLARO NXT TELECOMUNICACOES LTDA,NaN,TELEFONICA BRASIL S.A.,NaN,NaN,MOCN,Informe nº 111/2014-PRRE/CPRP/SPR/SCP,2841116,2014-02-18,ACÓRDÃO Nº 133/2014,2841116,2014-02-18,Processo iniciado no SICAP.
2,53500.017260/2015-34,2015-07-29,CONTRATO,NaN,NaN,TIM CELULAR S.A.,NaN,TELEFONICA BRASIL S.A.,NaN,OI MÓVEL S.A. - EM RECUPERAÇÃO JUDICIAL,MOCN,Informe nº 288/2015-PRRE/ORLE/ORER/CPRP/SPR/SO...,508651,2015-10-08,ACÓRDÃO Nº 555/2015,508651,2015-10-08,Processo iniciado no SICAP.
3,53500.001089/2014-61,2014-02-14,CONTRATO,NaN,40.432.544/0001-47,CLARO S.A.,NaN,TELEFONICA BRASIL S.A.,NaN,NaN,MOCN,Informe nº 189/2014-PRRE/CPRP/SPR/SCP,392130,2014-04-03,ACÓRDÃO Nº 199/2014,392130,2014-04-03,Processo iniciado no SICAP.
4,53500.020308/2014-19,2014-09-05,ADITIVO,53500.001089/2014-61,40.432.544/0001-47,CLARO S.A.,NaN,TELEFONICA BRASIL S.A.,NaN,NaN,MOCN,Informe nº 733/2014-CPRP/PRRE/SCP/SPR,392142,2014-12-05,ACÓRDÃO Nº 22/2015,392142,2014-12-05,Processo iniciado no SICAP.
5,53500.024608/2014-69,2014-10-22,ADITIVO,53500.001089/2014-61,40.432.544/0001-47,CLARO S.A.,NaN,TELEFONICA BRASIL S.A.,NaN,NaN,MOCN,Informe nº 733/2014-CPRP/PRRE/SCP/SPR,392142,2014-12-05,ACÓRDÃO Nº 22/2015,392142,2014-12-05,Processo iniciado no SICAP.
6,53500.002241/2016-94,2016-02-02,ADITIVO,53500.001089/2014-61,40.432.544/0001-47,CLARO S.A.,NaN,TELEFONICA BRASIL S.A.,NaN,NaN,MOCN,Informe nº 83/2016/SEI/CPRP/SCP,345635,2016-04-05,ACÓRDÃO Nº 194/2016,526288,2016-04-05,NaN
7,53500.002243/2016-83,2016-02-02,ADITIVO,53500.001089/2014-61,40.432.544/0001-47,CLARO S.A.,NaN,TELEFONICA BRASIL S.A.,NaN,NaN,MOCN,Informe nº 58/2016/SEI/CPRP/SCP,302087,2016-04-05,ACÓRDÃO Nº 194/2016,526288,2016-04-05,NaN
8,53500.010657/2016-86,2016-05-09,CONTRATO,NaN,66.970.229/0001-67,CLARO NXT TELECOMUNICACOES LTDA,NaN,TELEFONICA BRASIL S.A.,NaN,NaN,GWCN,Informe nº 179/2016/SEI/CPRP/SCP,504309,2016-06-28,ACÓRDÃO Nº 281/2016,720364,2016-06-28,NaN
9,53500.011812/2018-43,2018-03-29,ADITIVO,53500.031688/2012-47,NaN,TIM CELULAR S.A.,NaN,OI MÓVEL S.A. - EM RECUPERAÇÃO JUDICIAL,76.535.764/0001-43,OI S.A. - EM RECUPERACAO JUDICIAL,MOCN,Informe nº 180/2018/SEI/CPRP/SCP,2861810,2018-06-21,ACÓRDÃO Nº 409/2018,2991702,2018-06-21,NaN


#### Remoção de Duplicadas (novamente )

In [36]:
print(f'Tamanho dataset ANTES: {df_contratos_ran_sharing.shape}')
df_contratos_ran_sharing = df_contratos_ran_sharing.drop_duplicates()
print(f'Tamanho dataset DEPOIS: {df_contratos_ran_sharing.shape}')

Tamanho dataset ANTES: (16, 18)
Tamanho dataset DEPOIS: (16, 18)


#### Coluna de versionamento 

Vammos criar uma coluna para salvar as informações de versionamento de um contrato: 

* `Num Sequencia`, qual é a versão que aquele contrato está
* `final`, se aquela é a versão final ou não. 



In [80]:
df_contratos_ran_sharing = (
    df_contratos_ran_sharing
        .sort_values(by=['PROCESSO_ANATEL', 'PROTOCOLO_DATA'], kind='stable')
)
df_contratos_ran_sharing['NUM_SEQUENCIA'] = df_contratos_ran_sharing.groupby(by='PROCESSO_ANATEL').cumcount() + 1
df_max_seq = (
    df_contratos_ran_sharing
        .groupby('PROCESSO_ANATEL')
        [['NUM_SEQUENCIA']]
        .max()
        .reset_index()
        .rename(columns={'NUM_SEQUENCIA': 'MAX_NUM_SEQ'})
)
df_contratos_ran_sharing = df_contratos_ran_sharing.merge(df_max_seq, on='PROCESSO_ANATEL', how='left')
df_contratos_ran_sharing['FINAL']  = df_contratos_ran_sharing['NUM_SEQUENCIA'] == df_contratos_ran_sharing['MAX_NUM_SEQ']
df_contratos_ran_sharing = df_contratos_ran_sharing.drop(columns=['MAX_NUM_SEQ'])
df_contratos_ran_sharing

,PROCESSO_ANATEL,PROTOCOLO_DATA,ACORDO_TIPO,PROCESSO_ORIGEM,PRESTADORA_1_CNPJ,PRESTADORA_1,PRESTADORA_2_CNPJ,PRESTADORA_2,PRESTADORA_3_CNPJ,PRESTADORA_3,ACORDO_TECNOLOGIA,INFORME,INFORME_SEI,INFORME_DATA,ACORDAO,ACORDAO_SEI,ACORDAO_DATA,OBSERVACAO,NUM_SEQUENCIA,FINAL
0,53500.000608/2020-11,2020-01-07,CONTRATO,NaN,NaN,TELEFONICA BRASIL S.A.,NaN,TIM S A,NaN,NaN,GWCN/MOCN,Informe nº 206/2020/CPRP/SCP,5438699,2020-04-17,ACÓRDÃO Nº 209/2020,5503736,2020-04-17,NaN,1,True
1,53500.001089/2014-61,2014-02-14,CONTRATO,NaN,40.432.544/0001-47,CLARO S.A.,NaN,TELEFONICA BRASIL S.A.,NaN,NaN,MOCN,Informe nº 189/2014-PRRE/CPRP/SPR/SCP,392130,2014-04-03,ACÓRDÃO Nº 199/2014,392130,2014-04-03,Processo iniciado no SICAP.,1,True
2,53500.001164/2021-12,2021-01-08,CONTRATO,53500.001164/2021-12,NaN,TELEFONICA BRASIL S.A.,40.432.544/0001-47,CLARO S.A.,NaN,NaN,MOCN,Informe nº 68/2021/CPRP/SCP,6700033,2021-04-09,ACÓRDÃO Nº 286/2021,7295234,2021-04-09,NaN,1,False
3,53500.001164/2021-12,2021-03-15,ADITIVO,53500.001164/2021-12,NaN,TELEFONICA BRASIL S.A.,40.432.544/0001-47,CLARO S.A.,NaN,NaN,MOCN,Informe nº 68/2021/CPRP/SCP,6700033,2021-04-09,ACÓRDÃO Nº 286/2021,7295234,2021-04-09,NaN,2,True
4,53500.001341/2014-31,2014-01-09,CONTRATO,NaN,66.970.229/0001-67,CLARO NXT TELECOMUNICACOES LTDA,NaN,TELEFONICA BRASIL S.A.,NaN,NaN,MOCN,Informe nº 111/2014-PRRE/CPRP/SPR/SCP,2841116,2014-02-18,ACÓRDÃO Nº 133/2014,2841116,2014-02-18,Processo iniciado no SICAP.,1,True
5,53500.002241/2016-94,2016-02-02,ADITIVO,53500.001089/2014-61,40.432.544/0001-47,CLARO S.A.,NaN,TELEFONICA BRASIL S.A.,NaN,NaN,MOCN,Informe nº 83/2016/SEI/CPRP/SCP,345635,2016-04-05,ACÓRDÃO Nº 194/2016,526288,2016-04-05,NaN,1,True
6,53500.002243/2016-83,2016-02-02,ADITIVO,53500.001089/2014-61,40.432.544/0001-47,CLARO S.A.,NaN,TELEFONICA BRASIL S.A.,NaN,NaN,MOCN,Informe nº 58/2016/SEI/CPRP/SCP,302087,2016-04-05,ACÓRDÃO Nº 194/2016,526288,2016-04-05,NaN,1,True
7,53500.010657/2016-86,2016-05-09,CONTRATO,NaN,66.970.229/0001-67,CLARO NXT TELECOMUNICACOES LTDA,NaN,TELEFONICA BRASIL S.A.,NaN,NaN,GWCN,Informe nº 179/2016/SEI/CPRP/SCP,504309,2016-06-28,ACÓRDÃO Nº 281/2016,720364,2016-06-28,NaN,1,True
8,53500.011812/2018-43,2018-03-29,ADITIVO,53500.031688/2012-47,NaN,TIM CELULAR S.A.,NaN,OI MÓVEL S.A. - EM RECUPERAÇÃO JUDICIAL,76.535.764/0001-43,OI S.A. - EM RECUPERACAO JUDICIAL,MOCN,Informe nº 180/2018/SEI/CPRP/SCP,2861810,2018-06-21,ACÓRDÃO Nº 409/2018,2991702,2018-06-21,NaN,1,True
9,53500.017260/2015-34,2015-07-29,CONTRATO,NaN,NaN,TIM CELULAR S.A.,NaN,TELEFONICA BRASIL S.A.,NaN,OI MÓVEL S.A. - EM RECUPERAÇÃO JUDICIAL,MOCN,Informe nº 288/2015-PRRE/ORLE/ORER/CPRP/SPR/SO...,508651,2015-10-08,ACÓRDÃO Nº 555/2015,508651,2015-10-08,Processo iniciado no SICAP.,1,True


## Salvando Tabelas 

In [82]:
print('df_contratos_int:  ', df_contratos_int.shape)
df_contratos_int.to_csv('./Dados - Contratos Normalizados/contratos_int.csv', index=False)
print('df_contratos_comp:  ', df_contratos_comp.shape)
df_contratos_comp.to_csv('./Dados - Contratos Normalizados/contratos_comp.csv', index=False)
print('df_contratos_mvno:  ', df_contratos_mvno.shape)
df_contratos_mvno.to_csv('./Dados - Contratos Normalizados/contratos_mvno.csv', index=False)
print('df_contratos_ran_sharing:  ', df_contratos_ran_sharing.shape)
df_contratos_ran_sharing.to_csv('./Dados - Contratos Normalizados/contratos_ran_sharing.csv', index=False)

df_contratos_int:   (12364, 17)
df_contratos_comp:   (9617, 13)
df_contratos_mvno:   (341, 15)
df_contratos_ran_sharing:   (16, 20)
